## Chapter V. Task ##

#### 1.Download data from Don’tGetKicked competition. ####

In [3115]:
import pandas as pd
import numpy as np
from category_encoders import CountEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc
from sklearn.model_selection import GridSearchCV

In [3116]:
data = pd.read_csv('../datasets/training.csv')
data.PurchDate = pd.to_datetime(data['PurchDate'])
data.sort_values(by='PurchDate',ascending=True,inplace=True)

In [3117]:
data.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
32367,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,...,9906.00000,11657.00000,NaN,NaN,3453,80022,CO,6770.00000,0,1389
32384,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,...,5801.00000,6949.00000,NaN,NaN,22916,80022,CO,6160.00000,0,941
32385,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,...,4169.00000,5114.00000,NaN,NaN,3453,80022,CO,4250.00000,0,1155
32386,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,...,10438.00000,12158.00000,NaN,NaN,22916,80022,CO,8180.00000,0,1703
32387,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,...,4139.00000,5351.00000,NaN,NaN,22916,80022,CO,4900.00000,0,825


### 2.Design the train/validation/test split ###

In [3118]:
num = (len(data)//3)
X_train = data[0:num]
print(X_train.shape)
X_val = data[num:2*num]
print(X_val.shape)
X_test = data[2*num:len(data)]
print(X_test.shape)

(24327, 34)
(24327, 34)
(24329, 34)


In [3119]:
print(f'train max - {X_train["PurchDate"].max()}', f'val min - {X_val["PurchDate"].min()}', sep="\n")
print(f'val max - {X_val["PurchDate"].max()}', f'test min - {X_test["PurchDate"].min()}', sep="\n")

train max - 2009-09-15 00:00:00
val min - 2009-09-15 00:00:00
val max - 2010-05-14 00:00:00
test min - 2010-05-14 00:00:00


In [3120]:
X_train = data[data['PurchDate'] <= '2009-09-15']
X_val = data[(data['PurchDate'] > '2009-09-15') & (data['PurchDate'] <= '2010-05-14')]
X_test = data[data['PurchDate'] > '2010-05-14']

### 3.Counter Encoding ###

In [3123]:
for column in data.columns:
    precent = ((data[column].isna().sum()) / (data[column].shape[0]))*100
    if precent:
        print(f'{column} - {precent}')

X_train = X_train.drop(columns=['PRIMEUNIT','AUCGUART'],axis=1)
X_val = X_val.drop(columns=['PRIMEUNIT','AUCGUART'],axis=1)
X_test = X_test.drop(columns=['PRIMEUNIT','AUCGUART'],axis=1)

Trim - 3.2336297493936947
SubModel - 0.010961456777605743
Color - 0.010961456777605743
Transmission - 0.01233163887480646
WheelTypeID - 4.342107066029075
WheelType - 4.348957976515079
Nationality - 0.00685091048600359
Size - 0.00685091048600359
TopThreeAmericanName - 0.00685091048600359
MMRAcquisitionAuctionAveragePrice - 0.02466327774961292
MMRAcquisitionAuctionCleanPrice - 0.02466327774961292
MMRAcquisitionRetailAveragePrice - 0.02466327774961292
MMRAcquisitonRetailCleanPrice - 0.02466327774961292
MMRCurrentAuctionAveragePrice - 0.4316073606182262
MMRCurrentAuctionCleanPrice - 0.4316073606182262
MMRCurrentRetailAveragePrice - 0.4316073606182262
MMRCurrentRetailCleanPrice - 0.4316073606182262
PRIMEUNIT - 95.31534740967075
AUCGUART - 95.31534740967075


Как видим, колонки PRIMEUNIT, AUCGUART имеют очень малый процент качественной информации среди nan, поэтому я принял решение откинуть эти колонки

In [3124]:
def extract_date_features(df):
    df = df.copy()
    df['day'] = df['PurchDate'].dt.day
    df['month'] = df['PurchDate'].dt.month
    df = df.drop('PurchDate', axis=1)
    return df

X_train_object = extract_date_features(X_train)
X_val_object = extract_date_features(X_val)
X_test_object = extract_date_features(X_test)

In [3125]:
object_columns = []
for column in X_train.columns:
    if ((X_train[column].dtypes))=='object':
        object_columns.append(column)
object_columns

['Auction',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelType',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'VNST']

In [ ]:
encoder = CountEncoder(cols=object_columns,handle_missing='value')
encoder.fit(X_train_object)

X_train_enc = encoder.transform(X_train_object)
X_val_enc = encoder.transform(X_val_object)
X_test_enc = encoder.transform(X_test_object)

,verbose,0
,cols,"['Auction', 'Make', ...]"
,drop_invariant,False
,return_df,True
,handle_unknown,'value'
,handle_missing,'value'
,min_group_size,None
,combine_min_nan_groups,True
,min_group_name,None
,normalize,False


In [3128]:
train_mean = X_train_enc.mean()
X_train_enc = X_train_enc.fillna(train_mean)
X_val_enc = X_val_enc.fillna(train_mean)
X_test_enc = X_test_enc.fillna(train_mean)

In [3129]:
X_train_enc

,RefId,IsBadBuy,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost,day,month
32367,32389,0,14146,2007,2,2782,99,4547,284,3444,...,9906.00000,11657.00000,3453,80022,2025,6770.00000,0,1389,5,1
32384,32406,0,14146,2005,4,4274,309,265,46,5026,...,5801.00000,6949.00000,22916,80022,2025,6160.00000,0,941,5,1
32385,32407,0,14146,2004,5,4799,251,3444,1435,5026,...,4169.00000,5114.00000,3453,80022,2025,4250.00000,0,1155,5,1
32386,32408,0,14146,2006,3,5678,55,3000,177,4224,...,10438.00000,12158.00000,22916,80022,2025,8180.00000,0,1703,5,1
32387,32409,0,14146,2004,5,4274,956,265,58,1895,...,4139.00000,5351.00000,22916,80022,2025,4900.00000,0,825,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5580,5587,0,14146,2005,4,4274,161,99,77,1895,...,11255.00000,13041.00000,835,85040,2461,8705.00000,0,803,15,9
5579,5586,0,14146,2006,3,2782,124,317,141,2275,...,7414.00000,9470.00000,835,85040,2461,5590.00000,0,1373,15,9
63429,63460,0,14146,2004,5,5678,339,181,489,5026,...,6627.00000,7980.00000,18881,37210,613,7600.00000,0,1020,15,9
31497,31519,0,5247,2008,1,4799,280,3444,1435,5026,...,11452.00000,12105.00000,99761,91763,2558,8550.00000,0,834,15,9


In [3130]:
y_train = X_train_enc['IsBadBuy'].reset_index(drop=True)
X_train_enc.drop('IsBadBuy',axis=1,inplace=True)
y_val = X_val_enc['IsBadBuy'].reset_index(drop=True)
X_val_enc.drop('IsBadBuy',axis=1,inplace=True)
y_test = X_test_enc['IsBadBuy'].reset_index(drop=True)
X_test_enc.drop('IsBadBuy',axis=1,inplace=True)

In [3131]:
scaler = MinMaxScaler()
scaler.fit(X_train_enc)

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [3132]:
X_train = scaler.transform(X_train_enc)
X_val = scaler.transform(X_val_enc)
X_test = scaler.transform(X_test_enc)

In [3133]:
X_train = pd.DataFrame(X_train,columns=list(X_train_enc.columns))
X_val = pd.DataFrame(X_val,columns=list(X_train_enc.columns))
X_test = pd.DataFrame(X_test,columns=list(X_train_enc.columns))

### 4.Train LogisticRegression, GaussianNB, KNN ###

In [3135]:
def gini_score(y_score,y_true):
    auc = roc_auc_score(y_true,y_score)
    gini = 2 * auc - 1
    return gini

In [3136]:
Logreg = LogisticRegression()
Logreg.fit(X_train,y_train)

pred_proba_logreg = Logreg.predict_proba(X_val)[:,1]
gini_score_log_sk = gini_score(pred_proba_logreg,y_val)

In [3137]:
Gaus = GaussianNB()
Gaus.fit(X_train,y_train)

pred_proba_gaus = Gaus.predict_proba(X_val)[:,1]
gini_score_naive_sk = gini_score(pred_proba_gaus,y_val)

In [3138]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train,y_train)
pred_proba_knn = knn.predict_proba(X_val)[:,1]
gini_score_knn_sk = gini_score(pred_proba_knn,y_val)

In [3139]:
print(f'{gini_score_log_sk} - logisticregr,\n{gini_score_naive_sk} - GaussianNB,\n{gini_score_knn_sk} - knn')

0.44142029695444185 - logisticregr,
0.44160899971882794 - GaussianNB,
0.3021983195083724 - knn


### Анализ полученных метрик
Лучший результат по Gini нам выдала __Логистическая регрессия__.  

Оно не удивительно, логистическая регрессия хороша работает с линейно разделимыми данными и не боится шума, как KNN.  
Knn, в свою очередь, страдает от проклятия размерности, ему тяжело достигать высокие метрики при большом количестве признаков.  
Если говорить про наивный байс, то тут уже не совсем все однозначно.  
Как мы видим, наивный байес дал неплохую оценку, но почему так?  
В условия Байеса входит независимость и нормальность, но почему тогда при этом наша логистическая регрессия имеет высокую метрику?  
Мы же сказали, что она хорошо работает с линейно зависимыми признаками, а байес наоборот.  
Я считаю, что причина этого в том, что данные частично удовлетворяют обоим предложением.  
Особенно, после энкодинга признаки могут быть слабо коррелированными, либо наоборот.

### 5.Implement Gini score calculation ###

In [3140]:
def auc_roc(y_proba, y_true):
    import numpy as np

    y_true = np.array(y_true)
    y_proba = np.array(y_proba)

    desc_score_indices = np.argsort(y_proba)[::-1]
    y_true_sorted = y_true[desc_score_indices]

    pos = np.sum(y_true == 1)
    neg = np.sum(y_true == 0)

    count_pairs = 0
    count_y_bigger = 0

    for i in range(len(y_true_sorted)):
        if y_true_sorted[i] == 1:
            count_y_bigger += 1
        else:
            count_pairs += count_y_bigger
    auc = count_pairs / (pos * neg)

    return auc

In [3141]:
def gini_score_my(auc_roc):
    gini = 2 * auc_roc - 1
    return gini

In [3142]:
auc_roc_logreg = auc_roc(pred_proba_logreg,y_val)
gini_score_log_my = gini_score_my(auc_roc_logreg)

In [3143]:
auc_roc_naive = auc_roc(pred_proba_gaus,y_val)
gini_score_naive_my = gini_score_my(auc_roc_naive)

In [3144]:
auc_roc_knn = auc_roc(pred_proba_knn,y_val)
gini_score_knn_my = gini_score_my(auc_roc_knn)

In [3145]:
print(f'{gini_score_log_sk} - logisticregr,\n{gini_score_naive_sk} - GaussianNB,\n{gini_score_knn_sk} - knn')
print()
print(f'{gini_score_log_my} - logisticregr,\n{gini_score_naive_my} - GaussianNB,\n{gini_score_knn_my} - knn')

0.44142029695444185 - logisticregr,
0.44160899971882794 - GaussianNB,
0.3021983195083724 - knn

0.44142029695444185 - logisticregr,
0.44160895508362397 - GaussianNB,
0.2896533127162333 - knn


### 6.Implement your own versions ###

In [3146]:
class MyLogisticRegression:
    def __init__(self,learn_rate = 0.01,n_epochs=1000,random_state=None):
        self.learn_rate = learn_rate
        self.n_epochs = n_epochs
        self.random_state = random_state

    def sigmoida(self,x):
        return 1 / (1+np.exp(-x))

    def fit(self,X,y):
        X = np.array(X)
        y = np.array(y).flatten()

        X = np.c_[np.ones(X.shape[0]),X]
        n_samples = X.shape[0]
        n_features = X.shape[1]

        if self.random_state is not None:
            np.random.seed(self.random_state)
        self.w = np.zeros(n_features)

        for _ in range(self.n_epochs):
            idx = np.random.randint(0, n_samples)
            X_idx = X[idx]
            y_idx = y[idx]
            pred = np.dot(X_idx,self.w)
            y_proba = self.sigmoida(pred)
            grad = (y_proba - y_idx) * X_idx
            
            self.w = self.w - self.learn_rate * grad
        return self
    
    def predict_proba(self, X_test):
        X = np.array(X_test)
        X = np.c_[np.ones(X.shape[0]), X]
        return self.sigmoida(X @ self.w)
    
    def predict(self,X_test):
        X = np.array(X_test)
        X = np.c_[np.ones(X.shape[0]), X]
        y_proba = self.sigmoida(self.w, X)
        return np.where(y_proba >= 0.5,1,0)

In [3147]:
my_model = MyLogisticRegression(learn_rate = 0.01,n_epochs=10000,random_state=21)
my_model.fit(X_train,y_train)

In [3148]:
y_pred_logistic_my = my_model.predict_proba(X_val)
gini_score_log_my = gini_score(y_pred_logistic_my,y_val)

In [3149]:
class MyGaussianNaiveBayes:
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)

        self.classes_, counts = np.unique(y, return_counts=True)
        self.priors = counts / len(y)
        self.n_classes = len(self.classes_)

        self.means_ = np.array([np.mean(X[y == c], axis=0) for c in self.classes_])
        self.stds_ = np.array([np.std(X[y == c], axis=0) for c in self.classes_])
        self.stds_ = np.clip(self.stds_, 1e-9, None) 

    def formula(self, x, mean, std):
        return -0.5 * (np.log(2 * np.pi) + 2 * np.log(std) + ((x - mean) / std) ** 2)

    def predict_proba(self, X_test):
        X = np.asarray(X_test, dtype=np.float64)
        n_samples = X.shape[0]
        log_posteriors = np.zeros((n_samples, self.n_classes))

        for i in range(n_samples):
            x = X[i]
            for k in range(self.n_classes):
                log_likelihood = np.sum(self.formula(x, self.means_[k], self.stds_[k]))
                log_posteriors[i, k] = np.log(self.priors[k]) + log_likelihood

        log_posteriors -= np.max(log_posteriors, axis=1, keepdims=True)
        prob = np.exp(log_posteriors)
        prob /= np.sum(prob, axis=1, keepdims=True)
        return prob

    def predict(self, X_test):
        return self.classes_[np.argmax(self.predict_proba(X_test), axis=1)]

In [3150]:
my_model = MyGaussianNaiveBayes()
my_model.fit(X_train,y_train)

In [3151]:
y_pred_naive_my = my_model.predict_proba(X_val)
gini_score_naive_my = gini_score(y_pred_naive_my[:,1],y_val)

In [3152]:
class MyKNN:
    def __init__(self,k_neigbours):
        self.k_neighbours = k_neigbours

    def fit(self,X,y):
        self.X = np.asarray(X, dtype=np.float64)
        self.y = np.asarray(y)


    def predict_proba(self,X_test):
        X= np.asarray(X_test, dtype=np.float64)
        y_proba_list = []
        for idx,_ in enumerate(X):
            test_dot = X[idx]
            distances = []
            targets = []
            for i,train_dot in enumerate(self.X):
                dist = np.linalg.norm(train_dot - test_dot, ord=2)
                distances.append(dist)
                targets.append(self.y[i])
            distances = np.array(distances)
            targets = np.array(targets)

            k_top3_indices = np.argsort(distances)[:self.k_neighbours]
            k_top3_targets = targets[k_top3_indices]
            y_proba = np.sum(k_top3_targets) / self.k_neighbours

            y_proba_list.append(y_proba)


        return np.array(y_proba_list)
    
    def predict(self,X_test):
        proba = self.predict(X_test)
        return (proba >= 0.5).astype(int)


In [3153]:
model = MyKNN(k_neigbours=5)
model.fit(X_train,y_train)

In [3154]:
y_pred_knn_my = model.predict_proba(X_val[:5000])
gini_score_knn_my = gini_score(y_pred_knn_my,y_val[:5000])

In [3155]:
print('sklearn:')
print(f'{gini_score_log_sk} - logisticregr,\n{gini_score_naive_sk} - GaussianNB,\n{gini_score_knn_sk} - knn')
print('my:')
print(f'{gini_score_log_my} - logisticregr,\n{gini_score_naive_my} - GaussianNB,\n{gini_score_knn_my} - knn')

sklearn:
0.44142029695444185 - logisticregr,
0.44160899971882794 - GaussianNB,
0.3021983195083724 - knn
my:
0.42435009878217 - logisticregr,
0.4250860291483951 - GaussianNB,
0.3351548953626038 - knn


Собсна, у нас получились очень похожие метрики, но все равно есть небольшие несходности, причина которых может быть различие в гиперпараметрах.

### 7. Try to create non-linear features ###

In [3156]:
def add_extra_features_grouped(df):
    group_cols = ['Auction', 'Make', 'Nationality', 'Size', 'VNST']
    num_cols = ['VehBCost', 'WarrantyCost', 'VehicleAge']

    for gc in group_cols:
        for nc in num_cols:
            df[f'{gc}Mean{nc}'] = df[gc].map(df.groupby(gc)[nc].mean())
    return df

X_train = add_extra_features_grouped(X_train)
X_val = add_extra_features_grouped(X_val)
X_test = add_extra_features_grouped(X_test)

Я поэксперементировал с добавлениям фич и получил небольшие вывод:  
Метрика начинает повышаться только после добавления большого количества новых фич (10+), но когда добавляешь мало, то метрика почти никак не меняется

In [3157]:
def add_extra_features(X):
    X = X.copy()
    X['WarrantyToCost'] = X['WarrantyCost'] / (X['VehBCost'] + 1e-6)
    X['cost_to_retail_ratio'] = X['VehBCost'] / (X['MMRCurrentRetailAveragePrice'] + 1e-6)
    return X

X_train = add_extra_features(X_train) 
X_val = add_extra_features(X_val)
X_test = add_extra_features(X_test)

In [3158]:
encoder = CountEncoder(cols=object_columns,handle_missing='value')
encoder.fit(X_train)

,verbose,0
,cols,"['Auction', 'Make', ...]"
,drop_invariant,False
,return_df,True
,handle_unknown,'value'
,handle_missing,'value'
,min_group_size,None
,combine_min_nan_groups,True
,min_group_name,None
,normalize,False


In [3159]:
X_train = encoder.transform(X_train)
X_val = encoder.transform(X_val)
X_test = encoder.transform(X_test)

In [3160]:
train_mean = X_train.mean()
X_train_new = X_train.fillna(train_mean)
X_val_new = X_val.fillna(train_mean)
X_test_new = X_test.fillna(train_mean)

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_train_new)

X_train_scaled = scaler.transform(X_train_new)
X_val_scaled = scaler.transform(X_val_new)
X_test_scaled = scaler.transform(X_test_new)

X_train = X_train_scaled
X_val = X_val_scaled
X_test = X_test_scaled

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [ ]:
Logreg = LogisticRegression(max_iter=1000,random_state=21)
Logreg.fit(X_train,y_train)

pred_proba_logreg_new = Logreg.predict_proba(X_val)[:,1]
gini_score_log_sk_new = gini_score(pred_proba_logreg_new,y_val)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,21
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
Naive = GaussianNB()
Naive.fit(X_train,y_train)

pred_proba_gaus_new = Naive.predict_proba(X_val)[:,1]
gini_score_naive_sk_new = gini_score(pred_proba_gaus_new,y_val)

,priors,None
,var_smoothing,1e-09


In [ ]:
Knn_new = KNeighborsClassifier()
Knn_new.fit(X_train,y_train)

pred_proba_knn_new = Knn_new.predict_proba(X_val)[:,1]
gini_score_knn_sk_new = gini_score(pred_proba_knn_new,y_val)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [3169]:
print('sklearn:')
print(f'{gini_score_log_sk} - logisticregr,\n{gini_score_naive_sk} - GaussianNB,\n{gini_score_knn_sk} - knn')
print('my:')
print(f'{gini_score_log_sk_new} - logisticregr,\n{gini_score_naive_sk_new} - GaussianNB,\n{gini_score_knn_sk_new} - knn')

sklearn:
0.44142029695444185 - logisticregr,
0.44160899971882794 - GaussianNB,
0.3021983195083724 - knn
my:
0.44862435937695344 - logisticregr,
0.34990946939127476 - GaussianNB,
0.26087632712736375 - knn


появились небольшие улучшения в модели логистической регрессии, при это метрика на гауссовском наивном классификаторе упала.  
Оно и понятно, потому что новые признаки имеют сложные, не совсем гауссовские распределения.  
Если говорить насчет knn, то ожидаемо, что метрика упадет из-за увелечения размерности. 

### 8.Determine the best features ###

In [3170]:
features_names123 = X_train_new.columns.tolist()

In [3171]:
coefs_np = np.array(Logreg.coef_)[0]

coefs = pd.DataFrame({
    'ferature' : features_names123,
    'coefs' : abs(coefs_np)
}).sort_values(by='coefs',ascending=False)

In [3172]:
coefs

,ferature,coefs
11,WheelType,3.51434
27,VehBCost,3.04055
10,WheelTypeID,1.77251
39,NationalityMeanWarrantyCost,1.08655
46,VNSTMeanVehicleAge,1.06528
12,VehOdo,0.99207
3,VehicleAge,0.97346
23,MMRCurrentRetailCleanPrice,0.92116
24,BYRNO,0.89194
29,WarrantyCost,0.88494


In [ ]:
little_features = coefs[:25]['ferature'].tolist()
little_features

In [3174]:
X_train = pd.DataFrame(X_train,columns=list(X_train_new.columns))
X_val = pd.DataFrame(X_val,columns=list(X_train_new.columns))
X_test = pd.DataFrame(X_test,columns=list(X_train_new.columns))

In [3175]:
X_train_little = X_train[little_features]
X_val_little = X_val[little_features]
X_test_little = X_test[little_features]

In [3176]:
model = LogisticRegression()
model.fit(X_train_little,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [3177]:
y_pred_logreg = model.predict_proba(X_val_little)[:,1]
gini_score_logreg = gini_score(y_pred_logreg,y_val)
gini_score_logreg

0.44555857550733635

In [3178]:
model = LogisticRegression(solver='liblinear',penalty='l1',C=0.01)
model.fit(X_train_scaled,y_train)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,0.01
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [3179]:
y_pred_logreg_l1 = model.predict_proba(X_val_scaled)[:,1]
gini_score_logreg = gini_score(y_pred_logreg_l1,y_val)
gini_score_logreg

0.4658796002268777

### 9. Select your best model ###

In [3180]:
param_grid = {'C' : np.logspace(-4,2,50)}
model = LogisticRegression(solver='liblinear',penalty='l1')
grid = GridSearchCV(model, param_grid, scoring='roc_auc')
grid.fit(X_train_scaled,y_train)

,estimator,LogisticRegre...r='liblinear')
,param_grid,{'C': array([ 0.00...100. ])}
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l1'


In [3181]:
print(f'Best C = {grid.best_params_['C']}')
print(f'Best score = {2*grid.best_score_ - 1}')

Best C = 0.6250551925273969
Best score = 0.4824930048476126


В нашем случае, конечно же, будет значение числа С, то бишь насколько сильно наша регуляризация штрафут за высокие веса.  
Если говорить про другие модели, то для KNN будут количество соседей и метод измерения расстояния точек.

### 10. Check the Gini scores on all three datasets ###

In [ ]:
best_model = LogisticRegression(solver='liblinear',penalty='l1',C=grid.best_params_['C'])
best_model.fit(X_train_scaled,y_train)

y_pred_train = best_model.predict_proba(X_train_scaled)[:,1]
y_pred_val = best_model.predict_proba(X_val_scaled)[:,1]
y_pred_test = best_model.predict_proba(X_test_scaled)[:,1]

,penalty,'l1'
,dual,False
,tol,0.0001
,C,np.float64(0.6250551925273969)
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [3184]:
gini_score_train = gini_score(y_pred_train,y_train)
gini_score_val = gini_score(y_pred_val,y_val)
gini_score_test = gini_score(y_pred_test,y_test)

In [3185]:
print(f'All metrics on 3 datasets: \n {gini_score_train} - train\n {gini_score_val} - validation\n {gini_score_test} - test')

All metrics on 3 datasets: 
 0.5023934567292805 - train
 0.44395025007765776 - validation
 0.4838089147108977 - test


Я считаю, что нет сильных доказательств, чтобы говорить, что модель переобучилась. \
Метрики на train и validation достаточно близки, хотя и остается небольшой разрыв.  \
С test-выборкой все окей, метрика на ней почти такая же, как и на train 

### 11. Implement calculation of Recall, Precision, F1 score and AUC PR metrics. ###

In [3186]:
def calculate_metrics(y_true, y_pred_proba):
    y_true = np.asarray(y_true)
    y_pred_proba = np.asarray(y_pred_proba)
    
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    tp = np.sum((y_true == 1)&(y_pred == 1))
    fp = np.sum((y_true == 0)&(y_pred == 1))
    fn = np.sum((y_true == 1)&(y_pred == 0))
    
    recall = tp/(tp + fn)
    precision = tp/(tp + fp)
    f1 = 2 * (precision * recall)/(precision + recall)

    n_pos = np.sum(y_true)
    desc_idx = np.argsort(-y_pred_proba, kind="mergesort")
    y_true_sorted = y_true[desc_idx]
    
    # Накопленные tp и fp
    tp_cum = np.cumsum(y_true_sorted)
    fp_cum = np.cumsum(1 - y_true_sorted)
    precision_curve = tp_cum / (tp_cum + fp_cum)
    recall_curve = tp_cum / n_pos
    
    #начальная точка
    precision_curve = np.concatenate([[1.0], precision_curve])
    recall_curve = np.concatenate([[0.0], recall_curve])

    auc_pr = np.sum(precision_curve[1:] * (recall_curve[1:] - recall_curve[:-1]))

    return {
        'recall': float(recall),
        'precision': float(precision),
        'f1': float(f1),
        'auc_pr': float(auc_pr)
    }

In [3187]:
def check_metrics(model,X_train,y_train,X_test,y_test):
    model.fit(X_train,y_train)
    y_predict = model.predict_proba(X_test)[:,1]
    print(calculate_metrics(y_test,y_predict))

In [3188]:
model1 = LogisticRegression(solver='liblinear',penalty='l1',C=grid.best_params_['C'])
model2 = GaussianNB()
model3 = KNeighborsClassifier()

In [3189]:
print('LogisticRegression:')
check_metrics(model1,X_train_scaled,y_train,X_test_scaled,y_test)
print('GaussianNaiveBayes:')
check_metrics(model2,X_train_scaled,y_train,X_test_scaled,y_test)
print('KNeighborsClassifier:')
check_metrics(model3,X_train_scaled,y_train,X_test_scaled,y_test)

LogisticRegression:
{'recall': 0.23592135954681773, 'precision': 0.7814569536423841, 'f1': 0.3624264141284873, 'auc_pr': 0.42933608696281744}
GaussianNaiveBayes:
{'recall': 0.00433188937020993, 'precision': 0.7647058823529411, 'f1': 0.008614976805831677, 'auc_pr': 0.34411287410375685}
KNeighborsClassifier:
{'recall': 0.16727757414195268, 'precision': 0.606280193236715, 'f1': 0.2622094541655785, 'auc_pr': 0.3202349847609953}


### 12. Which hard label metric do you prefer for the task of detecting "lemon" cars?

Я считаю, что в нашем задании важнее всего получить высокий __recall__, чтобы быть уверенным, что большая часть настоящих лимонов будет найдено.